# At-20-cm cohort — unimodal vs multimodal comparison

Plots AUC across four arms on the 828-patient matched cohort (patients with both H&E at-20-cm AND RNA-seq at-20-cm). This removes the biopsy-site confound present in all-colon-sites experiments.

**Arms shown**
| Arm | Model | Features |
|-----|-------|----------|
| Unimodal image | prism2_base | Virchow2 base, 2 560-d |
| Unimodal image features | prism2_diagnostic | Virchow2 diagnostic, 3 072-d |
| Unimodal RNA | VST | CombatSeq + VST gene expression |
| Multimodal (best) | RNA + prism2_base, raw concat | Concat of both, no compression |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

RESULTS_DIR = ('/home/jovyan/kgbk271-ibd-volume/training/cd_vs_uc'
               '/08_09_at20cm_site_controlled/results')
OUT_DIR     = ('/home/jovyan/kgbk271-ibd-volume/training/cd_vs_uc'
               '/08_09_at20cm_site_controlled/reports')

In [ ]:
# ── Arm definitions ──────────────────────────────────────────────────────────
# strategy key  :  (display label, categorical colour slot)
# Display order: unimodal imaging → unimodal RNA → multimodal
# Colour: 4 categorical slots from palette.md (adjacent-validated, light mode)
#   slot 1 blue   #2a78d6  — imaging base
#   slot 2 orange #eb6834  — imaging features
#   slot 3 aqua   #1baf7a  — RNA
#   slot 4 yellow #eda100  — multimodal  (direct labels mandatory per palette.md)

ARMS = [
    ('img_base_20cm',       'Imaging base\n(prism2_base, 2 560-d)',       '#2a78d6'),
    ('img_diagnostic_20cm', 'Imaging features\n(prism2_diag., 3 072-d)',  '#eb6834'),
    ('rna_20cm',            'RNA only\n(VST)',                            '#1baf7a'),
    ('concat_raw_20cm',     'RNA + Imaging\n(concat raw, best)',          '#eda100'),
]

# chart chrome — palette.md
SURF  = '#fcfcfb'
INK   = '#0b0b0b'
INK2  = '#52514e'
MUTED = '#898781'
GRID  = '#e1e0d9'

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
fold_df = pd.read_csv(os.path.join(RESULTS_DIR, 'at20cm_fold_metrics.csv'))

strategy_keys = [a[0] for a in ARMS]
fold_df = fold_df[fold_df['strategy'].isin(strategy_keys)]

# per-arm summary
summary = (
    fold_df.groupby('strategy')['auc']
    .agg(mean_auc='mean', std_auc='std')
    .reset_index()
)
summary['std_auc'] = summary['std_auc'].fillna(0)

print(summary.to_string(index=False))

In [ ]:
# ── Build figure ──────────────────────────────────────────────────────────────
# Form: horizontal bar chart — comparing magnitude across 4 distinct identities
# Colour: categorical (identity job), 4 slots, adjacent-form validated
# Error bars: ± 1 std across 5 folds
# Per-fold dots: 5 points per arm for transparency on variance
# Direct labels: mandatory for slot-4 yellow contrast (palette.md note)

fig, ax = plt.subplots(figsize=(9, 4.2), facecolor=SURF)
ax.set_facecolor(SURF)

# ── bars (bottom → top: img_base, img_diag, rna, multimodal) ─────────────────
n = len(ARMS)
ys = np.arange(n)               # y positions, 0 = bottom
BAR_HEIGHT = 0.48
X_BASELINE = 0.5                # chance level = x-axis origin

for i, (strat, label, colour) in enumerate(ARMS):
    row = summary[summary['strategy'] == strat].iloc[0]
    mean_auc = row['mean_auc']
    std_auc  = row['std_auc']
    bar_width = mean_auc - X_BASELINE

    # bar
    ax.barh(ys[i], bar_width, left=X_BASELINE, height=BAR_HEIGHT,
            color=colour, linewidth=0, zorder=2)

    # error bar (± std)
    ax.errorbar(mean_auc, ys[i], xerr=std_auc,
                fmt='none', color=INK2, linewidth=1.2, capsize=4, capthick=1.2,
                zorder=3)

    # per-fold AUC dots
    fold_aucs = fold_df[fold_df['strategy'] == strat]['auc'].values
    rng = np.random.default_rng(42)
    jitter = rng.uniform(-0.12, 0.12, len(fold_aucs))
    ax.scatter(fold_aucs, ys[i] + jitter, s=22, color='white',
               edgecolors=INK, linewidths=0.7, zorder=4, clip_on=False)

    # AUC direct label (mandatory: slot-4 yellow has low surface contrast)
    ax.text(mean_auc + std_auc + 0.008, ys[i],
            f'{mean_auc:.3f} ± {std_auc:.3f}',
            va='center', ha='left', fontsize=8.5, color=INK, zorder=5)

# ── chance reference line ─────────────────────────────────────────────────────
ax.axvline(X_BASELINE, color=MUTED, linewidth=0.8, linestyle='--', zorder=1)
ax.text(X_BASELINE + 0.002, -0.65, 'chance', fontsize=7, color=MUTED, va='top')

# ── axes ─────────────────────────────────────────────────────────────────────
ax.set_yticks(ys)
ax.set_yticklabels([a[1] for a in ARMS], fontsize=9.5, color=INK)
ax.set_xlim(X_BASELINE, 0.92)
ax.set_ylim(-0.75, n - 0.25)
ax.set_xlabel('AUC (5-fold CV, patient-level)', fontsize=9.5, color=INK2)
ax.set_title('At-20-cm site-controlled · 828 matched patients (H&E + RNA)',
             fontsize=11, fontweight='bold', color=INK, pad=10)

# recessive x-grid
ax.xaxis.grid(True, color=GRID, linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

# spines — keep left + bottom only
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color(GRID)
ax.spines['bottom'].set_color(GRID)
ax.tick_params(colors=INK2, length=3)

# ── modality group brackets ──────────────────────────────────────────────────
# a thin bracket on the right groups the two imaging-only arms
bracket_x = 0.915
for y_lo, y_hi, txt in [(0, 1, 'Imaging'), (2, 2, ''), (3, 3, '')]:
    pass  # optional: add if visual grouping is desired

fig.tight_layout()
plt.show()
print('Figure drawn.')

In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, 'at20cm_unimodal_vs_multimodal.pdf')
fig.savefig(out_path, dpi=200, bbox_inches='tight', facecolor=SURF)
print(f'Saved → {out_path}')